In [44]:
import re
'''Get target <c0*> tag'''
c0_raw = input("Copy all <c0*> info here:")

'''Set global list variables'''


Copy all <c0*> info here: <c03 id="aspace_5a6edfeacd3b98a66a8e8d7eb378c8b1" level="item"><did><unittitle>Songs of Bathsheba : oratorio for soprano, choir and orchestra</unittitle><unitid>CD .S558575 D2.1</unitid><unitid audience="internal" identifier="659761605" type="OCLC">659761605</unitid><origination audience="internal" label="Creator"><persname authfilenumber="no2005009548" role="cmp" rules="rda" source="lcnaf">Shohat, Gil, 1973-</persname></origination><origination audience="internal" label="Creator"><persname authfilenumber="n  83003275" role="lyr" rules="rda" source="lcnaf">Shifrah, S., 1931-</persname></origination><origination audience="internal" label="Creator"><persname authfilenumber="no2007055397" role="sng" rules="rda" source="lcnaf">Robinson, Twyla</persname></origination><origination audience="internal" label="Creator"><persname authfilenumber="n  80046709" role="trl" rules="rda" source="lcnaf">Kaufman, Shirley</persname></origination><origination audience="internal" l

'Set global list variables'

In [45]:
names_list = re.findall(r"""<origination.+?<\S+name.+?</\S+name>""", c0_raw)
persnames_list = re.findall(r"""<origination.+?<persname.+?</persname>""", c0_raw)
corpnames_list = re.findall(r"""<origination.+?<corpname.+?</corpname>""", c0_raw)
#TO DO: TROUBLESHOOT

print(names_list)
print("")
print(persnames_list)
print("")
print(corpnames_list)

['<origination audience="internal" label="Creator"><persname authfilenumber="no2005009548" role="cmp" rules="rda" source="lcnaf">Shohat, Gil, 1973-</persname>', '<origination audience="internal" label="Creator"><persname authfilenumber="n  83003275" role="lyr" rules="rda" source="lcnaf">Shifrah, S., 1931-</persname>', '<origination audience="internal" label="Creator"><persname authfilenumber="no2007055397" role="sng" rules="rda" source="lcnaf">Robinson, Twyla</persname>', '<origination audience="internal" label="Creator"><persname authfilenumber="n  80046709" role="trl" rules="rda" source="lcnaf">Kaufman, Shirley</persname>', '<origination audience="internal" label="Creator"><corpname authfilenumber="nr 94030426" role="prf" rules="rda" source="lcnaf">Milwaukee Symphony Chorus</corpname>', '<origination audience="internal" label="Creator"><persname authfilenumber="no 89014215" role="prf" rules="rda" source="lcnaf">Milwaukee Symphony Orchestra</persname>', '<origination audience="interna

In [119]:
def ead2marc_245():

    '''INDICATORS'''
    '''Indicator 1 is constant (1)'''
    '''Indicator 2 is constant (0)'''

    
    '''SUBFIELD A'''
    '''Get clean title'''
    title_ead = str(re.findall(r"<unittitle>.+?</unittitle>", c0_raw))
    title_clean = title_ead.replace("<unittitle>", "").replace("</unittitle>", "").replace("['", "").replace("']", "")
    a_245 = f"""<marc:subfield code="a">{title_clean}</marc:subfield>"""

    
    '''PRINT 245 FIELD'''
    print("""<marc:datafield tag="245" ind1="1" ind2="0">""")
    print(a_245)
    print("""</marc:datafield""")

ead2marc_245()

<marc:datafield tag="245" ind1="1" ind2="0">
<marc:subfield code="a">קטע, לקלרנית = Piece for clarinet</marc:subfield>
</marc:datafield


In [24]:
def ead2marc_264():
    '''Create a list of all unitdates'''
    unitdates_list = re.findall(r"""<unitdate.+?</unitdate>""", c0_raw)

    
    '''Complete INDICATORS through PRINT 264 Field for each unitdate'''
    for unitdate in unitdates_list:

        '''INDICATORS'''
        '''Indicator 1 is constant (blank)'''
    
        '''Indicator 2'''
        '''Get date label (datechar)'''
        datechar_ead = str(re.findall(r"""<unitdate datechar=.+?" """, unitdate))
        datechar_clean = datechar_ead.replace("<unitdate datechar=", "").replace(" ", "").replace("['", "").replace("']", "")
    
        '''Set indicator 2 based on datechar'''
        if datechar_clean in ['"broadcast"', '"publication"']:
            ind2_264 = 1
        elif datechar_clean in ['"issued"']:
            ind2_264 = 3
        elif datechar_clean in ['"copyright"']:
            ind2_264 = 4
        else: 
            #AKA elif datechar_clean in ['"creation"', '"deaccession"', '"digitized"', '"event"', '"existence"', '"modified"', '"other"', '"record keeping"', '"usage"']
            ind2_264 = 0
    
        
        '''SUBFIELD C'''
        '''Set date to read [not after dddd] if datechar is "deaccession"'''
        if datechar_clean in ['"deaccession"']:
            qualifier_open = "[not after "
            qualifier_close = "]"
        else:
            qualifier_open = ""
            qualifier_close = ""
    
        '''Get clean date'''
        date_cleanish = str(re.findall(r"<unitdate.+>.+?</unitdate>", unitdate)).replace("</unitdate>", "").replace("['", "").replace("']", "")
        date_clean = re.sub("<unitdate.+>","", date_cleanish)
        c_264 = f"""<marc:subfield code ="c">{qualifier_open}{date_clean}{qualifier_close}<marc:subfield>"""
    
        
        '''PRINT 264 FIELD'''
        print(f"""<marc:datafield tag="264" ind1=" " ind2="{ind2_264}">""")
        print (c_264)
        print("""</marc:datafield""")
        print("")

ead2marc_264()

<marc:datafield tag="264" ind1=" " ind2="0">
<marc:subfield code ="c">2005<marc:subfield>
</marc:datafield



In [38]:
def ead2marc_100_110():
    '''Determine whether first name is persname or corpname to determine if field is 100 or 110'''
    main_name_ead = str(names_list[0])
    
    if "<persname " in main_name_ead:
        field_100_or_110 = "100"
    elif "<corpname " in main_name_ead:
        field_100_or_110 = "110"
    ### TO DO: CREATE CASES FOR OTHER EAD NAME ELEMENTS (agencyname, famname, geogname, namegrp)
    
        
    '''INDICATORS'''
    '''Indicator 1'''
    datechar_ead = str(re.findall(r"""<unitdate datechar=.+?" """, unitdate))
    datechar_clean = datechar_ead.replace("<unitdate datechar=", "").replace(" ", "").replace("['", "").replace("']", "")
    
    '''Indicator 2 is constant (blank)'''

    '''PRINT 100/110 FIELD'''
    print(f"""<marc:datafield tag="{field_100_or_110}" ind1=" " ind2=" ">""")
    print ("")
    print("""</marc:datafield""")
    print("")


ead2marc_100_110()

<marc:datafield tag="100" ind1=" " ind2=" ">

</marc:datafield



In [ ]:
def ead2marc_700():
    '''Create a list of all names and a list of all persnames'''

    '''Complete INDICATORS through PRINT 700 Field for each persname'''
    for persname in persname_list:
        
        '''INDICATORS'''
        '''Indicator 1'''
        
    
        '''Indicator 2 is constant (blank)'''
    


ead2marc_100()